Importando as bibliotecas

In [1]:
import psycopg2
import os
import io
from dotenv import load_dotenv
import pandas as pd
import sqlalchemy

Definindo a conexão

In [2]:
load_dotenv(dotenv_path="../../.env")

host=os.getenv("host")
dbname=os.getenv("dbname")
user=os.getenv("user")
password=os.getenv("password")
port=os.getenv("port")

engine = sqlalchemy.create_engine(f'postgresql://{user}:{password}@{host}:{port}/{dbname}')


1) Qual é o faturamento total por categoria de produto?

In [3]:
query_1 = """
        SELECT categoria,
        sum(quantidade * preco_unitario) AS faturamento_por_categoria
        FROM case_1.vendas_silver
        GROUP BY categoria
        """
Q1 = pd.read_sql(query_1, engine)

Q1


,categoria,faturamento_por_categoria
0,calçados,223566.50
1,acessórios,141127.48
2,roupas,279925.60


2) Quais são as 5 cidades com maior volume de vendas (em R$)?

In [4]:
query_2 = """
        SELECT cidade,
        sum(quantidade * preco_unitario) AS faturamento_por_cidade
        FROM case_1.vendas_silver
        GROUP BY cidade
        ORDER BY faturamento_por_cidade DESC
        LIMIT 5
        """
Q2 = pd.read_sql(query_2, engine)

Q2

,cidade,faturamento_por_cidade
0,belo horizonte,159756.40
1,rio de janeiro,79513.70
2,fortaleza,67831.56
3,são paulo,59610.80
4,brasília,54547.80


3) Qual produto tem o maior ticket médio por unidade vendida?

In [5]:
query_3 = """
        SELECT produto,
        avg(preco_unitario) AS preco_medio
        FROM case_1.vendas_silver
        GROUP BY produto
        ORDER BY preco_medio DESC
        LIMIT 1
        """
Q3 = pd.read_sql(query_3, engine)

Q3

,produto,preco_medio
0,vestido floral,269.351667


4) Como evoluiu o faturamento mês a mês ao longo dos 2 anos?


In [6]:
query_4 = """
       WITH t1 AS (
       SELECT 
                TO_CHAR(data_venda, 'YYYY-MM') AS ano_mes,
                SUM(quantidade * preco_unitario) AS faturamento
        FROM case_1.vendas_silver
        GROUP BY ano_mes
       )
       SELECT t1.*, 
       SUM(faturamento) OVER (ORDER BY ano_mes) AS acumulado
       FROM t1
        """
Q4 = pd.read_sql(query_4, engine)

Q4

,ano_mes,faturamento,acumulado
0,2023-01,34998.30,34998.30
1,2023-02,22006.80,57005.10
2,2023-03,48742.20,105747.30
3,2023-04,26973.70,132721.00
4,2023-05,17625.60,150346.60
5,2023-06,32082.70,182429.30
6,2023-07,8631.60,191060.90
7,2023-08,12356.90,203417.80
8,2023-09,55222.32,258640.12
9,2023-10,19201.26,277841.38


5) Existe sazonalidade nas vendas? Compare o volume de cada trimestre.

In [7]:
query_5 = """
        WITH t1 AS (
        SELECT 
                TO_CHAR(data_venda, 'YYYY-MM') AS ano_mes,
                SUM(quantidade * preco_unitario) AS faturamento,
                EXTRACT(MONTH FROM data_venda) AS mes,
                CASE 
                        WHEN TO_CHAR(data_venda, 'YYYY') = '2023' THEN 1
                        WHEN TO_CHAR(data_venda, 'YYYY') = '2024' THEN 2
                END AS ano
        FROM case_1.vendas_silver
        GROUP BY ano_mes, mes, ano
        ),
        t2 AS (
        SELECT 
                t1.*,
                CASE 
                        WHEN mes BETWEEN 1 AND 3 THEN '1_trimestre'
                        WHEN mes BETWEEN 4 AND 6 THEN '2_trimestre'
                        WHEN mes BETWEEN 7 AND 9 THEN '3_trimestre'
                        WHEN mes BETWEEN 10 AND 12 THEN '4_trimestre' 
                END AS trimestre
        FROM t1
        ORDER BY ano_mes)

        SELECT t2.trimestre, sum(t2.faturamento), t2.ano
        FROM t2
        GROUP BY t2.trimestre, t2.ano
        ORDER BY t2.trimestre, t2.ano
        ;
        """
Q5 = pd.read_sql(query_5, engine)

Q5

,trimestre,sum,ano
0,1_trimestre,105747.30,1
1,1_trimestre,47681.40,2
2,2_trimestre,76682.00,1
3,2_trimestre,111694.40,2
4,3_trimestre,76210.82,1
5,3_trimestre,76001.00,2
6,4_trimestre,80615.16,1
7,4_trimestre,69987.50,2


In [9]:
Q1.to_sql("faturamento_categoria_gold", con=engine, schema="case_1", if_exists="replace")
Q2.to_sql("top5_cidades_vol_gold", con=engine, schema="case_1", if_exists="replace")
Q3.to_sql("maior_ticket_UV_gold", con=engine, schema="case_1", if_exists="replace")
Q4.to_sql("evolucao_faturamento_gold", con=engine, schema="case_1", if_exists="replace")
Q5.to_sql("sazonalidade", con=engine, schema="case_1", if_exists="replace")

8